In [1]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [2]:
torch.manual_seed(42)
np.random.seed(42)

In [3]:
print("--- Initializing ML Pipeline ---")

# 1. Load the pristine data from the hard drive
X = np.load('processed_data/X_normalized.npy')
Y_identity = np.load('processed_data/Y_identity.npy')
Y_utility = np.load('processed_data/Y_utility.npy')
Run_IDs = np.load('processed_data/Run_IDs.npy')

# Dynamically load the channel masks!
emotiv_indices = np.load('processed_data/emotiv_indices.npy').tolist()
muse_indices = np.load('processed_data/muse_indices.npy').tolist()

print(f"Data Loaded! Total Matrix Shape: {X.shape}")

assert X.shape[1] == 64 and X.shape[2] == 160, f"Unexpected shape: {X.shape}"
assert len(X) == len(Y_identity) == len(Y_utility) == len(Run_IDs), "Length mismatch!"
print(f"Verified Checkpoint: {X.shape[0]} epochs | 64 channels | 160 timepoints")

--- Initializing ML Pipeline ---
Data Loaded! Total Matrix Shape: (1800, 64, 160)
Verified Checkpoint: 1800 epochs | 64 channels | 160 timepoints


In [4]:
train_mask = (Run_IDs == 3) | (Run_IDs == 7)
test_mask = (Run_IDs == 11)

X_train, X_test = X[train_mask], X[test_mask]
Y_id_train, Y_id_test = Y_identity[train_mask], Y_identity[test_mask]
Y_util_train, Y_util_test = Y_utility[train_mask], Y_utility[test_mask]

rest_count = (Y_util_train == 0).sum()
move_count = (Y_util_train == 1).sum()
print(f"\nUtility class balance - Rest: {rest_count}, Move: {move_count}")

ratio = max(rest_count, move_count) / min(rest_count, move_count)
if ratio > 1.5:
    print(f"  WARNING: Class imbalance ratio {ratio:.2f}x - we will need weighted loss!")
else:
    print(f"  Class balance looks healthy.")

class EEGDataset(Dataset):
    def __init__(self, X_data, y_labels):
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_labels, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


Utility class balance - Rest: 600, Move: 600
  Class balance looks healthy.


In [5]:
batch_size = 32

# Identity Loaders
train_loader_id = DataLoader(EEGDataset(X_train, Y_id_train), batch_size=batch_size, shuffle=True)
test_loader_id = DataLoader(EEGDataset(X_test, Y_id_test), batch_size=batch_size, shuffle=False)

# Utility Loaders
train_loader_util = DataLoader(EEGDataset(X_train, Y_util_train), batch_size=batch_size, shuffle=True)
test_loader_util = DataLoader(EEGDataset(X_test, Y_util_test), batch_size=batch_size, shuffle=False)

print("\n--- PyTorch DataLoaders Ready ---")
print(f"Identity Model: Predicting {len(np.unique(Y_identity))} Subjects.")
print(f"Train size: {len(X_train)} epochs | Test size: {len(X_test)} epochs")
print(f"Utility Model: Predicting 2 States (Rest vs. Move).")


--- PyTorch DataLoaders Ready ---
Identity Model: Predicting 20 Subjects.
Train size: 1200 epochs | Test size: 600 epochs
Utility Model: Predicting 2 States (Rest vs. Move).


## CNN

In [6]:
class EEGCNN(nn.Module):
    def __init__(self, n_channels, n_classes):
        """
        A single, reusable CNN for both tasks and all channel conditions.
        
        Args:
            n_channels: 64 (full), 14 (Emotiv), or 4 (Muse)
            n_classes:  20 (identity task) or 2 (utility task)
        """
        super(EEGCNN, self).__init__()
        
        # --- Feature Extractor ---
        # Conv1d reads along the TIME dimension (160 timepoints)
        # Input shape: (batch, n_channels, 160)
        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(n_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),          # (batch, 32, 80)
            
            # Block 2
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),          # (batch, 64, 40)
            
            # Block 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),  # (batch, 128, 1) — collapses time dimension
        )
        
        # --- Classifier Head ---
        self.classifier = nn.Sequential(
            nn.Flatten(),             # (batch, 128)
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, n_classes)  # output sized to task
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
    def apply_channel_mask(X_batch, indices):
        """
        X_batch shape: (batch, 64, 160)
        returns shape: (batch, len(indices), 160)
        """
        return X_batch[:, indices, :]


# --- Verify all 6 model configurations ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}\n")

configs = [
    (64, "64-channel Full"),
    (14, "14-channel Emotiv"),
    (4,  "4-channel Muse"),
]

print("--- Architecture Verification ---")
for n_ch, label in configs:
    for n_cls, task in [(20, "Identity"), (2, "Utility")]:
        model = EEGCNN(n_channels=n_ch, n_classes=n_cls).to(device)
        dummy = torch.zeros(32, n_ch, 160).to(device)
        out = model(dummy)
        print(f"  [{label} | {task}] Input: {tuple(dummy.shape)} → Output: {tuple(out.shape)} ✓")

Training on: cpu

--- Architecture Verification ---
  [64-channel Full | Identity] Input: (32, 64, 160) → Output: (32, 20) ✓
  [64-channel Full | Utility] Input: (32, 64, 160) → Output: (32, 2) ✓
  [14-channel Emotiv | Identity] Input: (32, 14, 160) → Output: (32, 20) ✓
  [14-channel Emotiv | Utility] Input: (32, 14, 160) → Output: (32, 2) ✓
  [4-channel Muse | Identity] Input: (32, 4, 160) → Output: (32, 20) ✓
  [4-channel Muse | Utility] Input: (32, 4, 160) → Output: (32, 2) ✓


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

def apply_channel_mask(X_batch, indices):
    """
    Slices the 64-channel tensor down to the target hardware subset.
    If indices is None, it returns the full 64-channel baseline.
    """
    if indices is None:
        return X_batch
    return X_batch[:, indices, :]

def train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, epochs, channel_indices, device):
    """
    The master loop that trains the model and evaluates it on the test set.
    """
    model.to(device)
    
    # TEAMMATE FIX 2: Initialize avg_loss to prevent scope risk if epochs=0
    avg_loss = 0.0 
    
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        running_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Apply the privacy filter BEFORE the model sees the data
            X_batch = apply_channel_mask(X_batch, channel_indices)
            
            # Zero gradients, forward pass, backward pass, optimize
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        # Prevent division by zero just in case the dataloader is weird
        if len(train_loader) > 0:
            avg_loss = running_loss / len(train_loader)
        
        # --- QUICK VALIDATION CHECK (Every 5 Epochs) ---
        if (epoch + 1) % 5 == 0:
            model.eval() # Temporarily turn off training behaviors like Dropout
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for X_val, y_val in test_loader:
                    X_val, y_val = X_val.to(device), y_val.to(device)
                    X_val = apply_channel_mask(X_val, channel_indices)
                    
                    val_outputs = model(X_val)
                    _, val_predicted = torch.max(val_outputs.data, 1)
                    
                    val_total += y_val.size(0)
                    val_correct += (val_predicted == y_val).sum().item()
                    
            val_acc = 100 * val_correct / val_total
            print(f"  Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f} | Val Acc: {val_acc:.1f}%")
            
            # TEAMMATE FIX 1: Explicitly return to training mode
            model.train() 

    # --- FINAL EVALUATION PHASE ---
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            X_batch = apply_channel_mask(X_batch, channel_indices)
            
            outputs = model(X_batch)
            _, predicted = torch.max(outputs.data, 1)
            
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
            
    final_accuracy = 100 * correct / total
    
    # Returning a dictionary for clean data collection
    return {
        'accuracy': final_accuracy,
        'epochs_trained': epochs,
        'final_loss': avg_loss
    }

print("--- Fully Patched Training Engine Loaded ---")

--- Fully Patched Training Engine Loaded ---


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing Tradeoff Experiments on: {device}\n")

# TEAMMATE FIX 1: Robust Class Imbalance Handling
try:
    if ratio > 1.5:
        weight_move = rest_count / move_count
        utility_weights = torch.tensor([1.0, weight_move], dtype=torch.float32).to(device)
        criterion_util = nn.CrossEntropyLoss(weight=utility_weights)
        print(f"Weighted loss applied: move weight = {weight_move:.2f}")
    else:
        criterion_util = nn.CrossEntropyLoss()
        print("Standard loss applied: classes are balanced.")
except NameError:
    criterion_util = nn.CrossEntropyLoss()
    print("WARNING: Class balance unknown, using standard loss. Re-run Cell 1 to verify.")

criterion_id = nn.CrossEntropyLoss()

# TEAMMATE FIX 2: Bumping epochs to allow full convergence
epochs_to_train = 50 

# --- Define the 6 Configurations ---
experiments = [
    {"name": "Clinical Baseline (64 Channels)", "channels": 64, "indices": None},
    {"name": "Emotiv EPOC+ (14 Channels)", "channels": 14, "indices": emotiv_indices},
    {"name": "Muse Headband (4 Channels)", "channels": 4, "indices": muse_indices}
]

results = {}

print("\n==================================================")
print("STARTING PRIVACY-UTILITY TRADEOFF EVALUATION")
print("==================================================")

# TEAMMATE FIX 4: Establish the Random Chance Baselines
random_chance_id = 100 / 20  # 5.0% for 20 subjects
random_chance_util = 50.0    # ~50% for binary Rest vs. Move
print(f"Random Chance Baselines: Identity = {random_chance_id}%, Utility = {random_chance_util}%\n")

for exp in experiments:
    name = exp["name"]
    ch = exp["channels"]
    idx = exp["indices"]
    
    print(f"--- Testing Hardware: {name} ---")
    
    # TEAMMATE FIX 3: Reset seeds immediately before instantiating the models
    # This guarantees apples-to-apples weight initialization across all hardware tests
    torch.manual_seed(42)
    np.random.seed(42)
    
    # 1. Test the Privacy Threat (Identity Task)
    print("  [Task 1] Training Adversarial Identity Model (20 Subjects)...")
    model_id = EEGCNN(n_channels=ch, n_classes=20).to(device)
    optimizer_id = optim.Adam(model_id.parameters(), lr=0.001)
    
    res_id = train_and_evaluate(
        model=model_id, train_loader=train_loader_id, test_loader=test_loader_id, 
        criterion=criterion_id, optimizer=optimizer_id, epochs=epochs_to_train, 
        channel_indices=idx, device=device
    )
    
    # Reset seeds again for the Utility model to be perfectly mathematically isolated
    torch.manual_seed(42)
    np.random.seed(42)
    
    # 2. Test the Clinical Utility (Motor Movement Task)
    print("  [Task 2] Training Utility Task Model (Rest vs. Move)...")
    model_util = EEGCNN(n_channels=ch, n_classes=2).to(device)
    optimizer_util = optim.Adam(model_util.parameters(), lr=0.001)
    
    res_util = train_and_evaluate(
        model=model_util, train_loader=train_loader_util, test_loader=test_loader_util, 
        criterion=criterion_util, optimizer=optimizer_util, epochs=epochs_to_train, 
        channel_indices=idx, device=device
    )
    
    # Store results for final printout
    results[name] = {
        "Privacy_Risk": res_id['accuracy'],
        "Utility_Preserved": res_util['accuracy']
    }
    
    print(f"  Result -> Privacy Risk (ID Acc): {res_id['accuracy']:.1f}% | Utility (Task Acc): {res_util['accuracy']:.1f}%\n")

# --- Final Output for your Professor's Graph ---
print("==================================================")
print("FINAL TRADEOFF RESULTS (Copy these for your report)")
print("==================================================")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"   Adversary Identification Accuracy: {metrics['Privacy_Risk']:.2f}%")
    print(f"   Clinical Utility Accuracy:         {metrics['Utility_Preserved']:.2f}%")

Executing Tradeoff Experiments on: cpu

Standard loss applied: classes are balanced.

STARTING PRIVACY-UTILITY TRADEOFF EVALUATION
Random Chance Baselines: Identity = 5.0%, Utility = 50.0%

--- Testing Hardware: Clinical Baseline (64 Channels) ---
  [Task 1] Training Adversarial Identity Model (20 Subjects)...
  Epoch [5/50] Loss: 1.3265 | Val Acc: 63.3%
  Epoch [10/50] Loss: 0.5363 | Val Acc: 84.3%
  Epoch [15/50] Loss: 0.3049 | Val Acc: 78.5%
  Epoch [20/50] Loss: 0.2292 | Val Acc: 87.3%
  Epoch [25/50] Loss: 0.1497 | Val Acc: 93.7%
  Epoch [30/50] Loss: 0.1283 | Val Acc: 89.7%
  Epoch [35/50] Loss: 0.1148 | Val Acc: 93.7%
  Epoch [40/50] Loss: 0.0720 | Val Acc: 95.0%
  Epoch [45/50] Loss: 0.0866 | Val Acc: 91.7%
  Epoch [50/50] Loss: 0.0743 | Val Acc: 91.8%
  [Task 2] Training Utility Task Model (Rest vs. Move)...
  Epoch [5/50] Loss: 0.5721 | Val Acc: 59.5%
  Epoch [10/50] Loss: 0.4884 | Val Acc: 67.2%
  Epoch [15/50] Loss: 0.3774 | Val Acc: 68.3%
  Epoch [20/50] Loss: 0.3382 | Val

In [9]:
torch.manual_seed(42)
model_test = EEGCNN(n_channels=64, n_classes=20).to(device)
optimizer_test = optim.Adam(model_test.parameters(), lr=0.001)
res_test = train_and_evaluate(
    model=model_test, train_loader=train_loader_id, test_loader=test_loader_id,
    criterion=criterion_id, optimizer=optimizer_test, epochs=5,
    channel_indices=None, device=device
)
print(f"Smoke test passed: {res_test}")

  Epoch [5/5] Loss: 1.3265 | Val Acc: 63.3%
Smoke test passed: {'accuracy': 63.333333333333336, 'epochs_trained': 5, 'final_loss': 1.326481624653465}
